# [실습 1] 서울 열린데이터 광장 Open API 활용
- https://data.seoul.go.kr/
- 서울시 공공자전거 실시간 대여정보

In [1]:
# 라이브러리 불러오기
import urllib.request
import json
import pandas as pd

## 1.1. 서울시 공공자전거 실시간 대여정보 1,000건 가져오기

- 서울특별시 공공자전거 실시간 대여정보를 가져와 **bike** 데이터프레임을 만듭니다.
- 데이터셋은 대여소별 실시간 자전거 대여가능 건수, 거치율, 대여소 위치정보를 제공합니다.
- 호출시 시스템 부하로 한번에 최대 **1,000건**를 초과할수 없습니다.
- 우선 1,000개 행만 조회하세요.

In [3]:
# 인증키와  URL 주소
# https://data.seoul.go.kr/dataList/OA-15493/A/1/datasetView.do
# uv add python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()

key = os.getenv('bike_key')
res_type = 'json'
start = 1
end = 1000

url = f'http://openapi.seoul.go.kr:8088/{key}/{res_type}/bikeList/{start}/{end}'

print(url)


http://openapi.seoul.go.kr:8088/52795054456a657838324e4b62547a/json/bikeList/1/1000


In [3]:
response = urllib.request.urlopen(url)
json_str = response.read().decode('utf-8')

print(json_str)

{"rentBikeStatus":{"list_total_count":1000,"RESULT":{"CODE":"INFO-000","MESSAGE":"정상 처리되었습니다."},"row":[{"rackTotCnt":"15","stationName":"102. 망원역 1번출구 앞","parkingBikeTotCnt":"12","shared":"80","stationLatitude":"37.55564880","stationLongitude":"126.91062927","stationId":"ST-4"},{"rackTotCnt":"14","stationName":"103. 망원역 2번출구 앞","parkingBikeTotCnt":"21","shared":"150","stationLatitude":"37.55495071","stationLongitude":"126.91083527","stationId":"ST-5"},{"rackTotCnt":"13","stationName":"104. 합정역 1번출구 앞","parkingBikeTotCnt":"14","shared":"108","stationLatitude":"37.55073929","stationLongitude":"126.91508484","stationId":"ST-6"},{"rackTotCnt":"5","stationName":"105. 합정역 5번출구 앞","parkingBikeTotCnt":"3","shared":"60","stationLatitude":"37.55000687","stationLongitude":"126.91482544","stationId":"ST-7"},{"rackTotCnt":"12","stationName":"106. 합정역 7번출구 앞","parkingBikeTotCnt":"12","shared":"100","stationLatitude":"37.54864502","stationLongitude":"126.91282654","stationId":"ST-8"},{"rackTotCnt":"1

**데이터셋 정보**

- rackTotCnt: 거치대개수
- stationName: 대여소이름
- parkingBikeTotCnt: 자전거주차총건수
- shared: 거치율
- stationLatitude: 위도
- stationLongitude: 경도
- stationId: 대여소ID

## 1.2. 서울시 공공자전거 실시간 대여정보 모든 데이터 가져오기

- 위 실습 결과로 작성된 구문을 수정하여 모든 데이터를 가져와 **bike** 데이터프레임을 다시 만듭니다.
- while 문을 사용해 JSON 문자열을 연결 한 후 이후 과정을 진행합니다.

In [4]:
start = 1
end = 1000
result = []
while True:

  url = f'http://openapi.seoul.go.kr:8088/{key}/{res_type}/bikeList/{start}/{end}'
  response = urllib.request.urlopen(url)
  json_str = response.read().decode('utf-8')
  # print(json_str)
  json_object = json.loads(json_str)
  # bike = pd.json_normalize(json_object['rentBikeStatus']['row'])
  # result.append(bike)
  result.extend(json_object['rentBikeStatus']['row'])

  print(len(json_object['rentBikeStatus']['row']))

  # print(json_object['rentBikeStatus']['list_total_count'])
  if int(json_object['rentBikeStatus']['list_total_count']) < 1000:
    break

  start += 1000
  end += 1000
final = pd.json_normalize(result)
print(len(result))
print(final.info())

1000
1000
727
2727
<class 'pandas.DataFrame'>
RangeIndex: 2727 entries, 0 to 2726
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   rackTotCnt         2727 non-null   str  
 1   stationName        2727 non-null   str  
 2   parkingBikeTotCnt  2727 non-null   str  
 3   shared             2727 non-null   str  
 4   stationLatitude    2727 non-null   str  
 5   stationLongitude   2727 non-null   str  
 6   stationId          2727 non-null   str  
dtypes: str(7)
memory usage: 149.3 KB
None


In [5]:
loop = 0
result = []

while True:
  start = 1 + 1000 * loop # 1 1001 2001 ~
  end = 1000 + 1000 * loop # 1000 2000 3000

  url = f'http://openapi.seoul.go.kr:8088/{key}/{res_type}/bikeList/{start}/{end}'

  # 데이터 가져오기
  response = urllib.request.urlopen(url)
  json_str = response.read().decode('utf-8')

  # 딕셔너리로 변환
  json_object = json.loads(json_str)

  result += json_object['rentBikeStatus']['row']

  print(loop)

  print(len(result))
  print('=' * 40)
  if (len(result) % 1000 != 0):
    break
  loop += 1

0
1000
1
2000
2
2727


- 하위 데이터를 확인합니다.

In [ ]:
# json_object = json.loads(json_str)
# print(json_object)

# # result = json_object['rentBikeStatus']['row']

# # 데이터 플레임으로 변환
# bike = pd.json_normalize(json_object['rentBikeStatus']['row'])
# bike

{'rentBikeStatus': {'list_total_count': 727, 'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다.'}, 'row': [{'rackTotCnt': '10', 'stationName': '3912. 삼호아파트 앞', 'parkingBikeTotCnt': '8', 'shared': '80', 'stationLatitude': '37.49385834', 'stationLongitude': '126.85646820', 'stationId': 'ST-2855'}, {'rackTotCnt': '10', 'stationName': '3914. 신도림쌍용플래티넘노블아파트 앞', 'parkingBikeTotCnt': '17', 'shared': '170', 'stationLatitude': '37.50126266', 'stationLongitude': '126.89118195', 'stationId': 'ST-2892'}, {'rackTotCnt': '10', 'stationName': '3915. 동선아파트 앞', 'parkingBikeTotCnt': '14', 'shared': '140', 'stationLatitude': '37.49382782', 'stationLongitude': '126.83907318', 'stationId': 'ST-2949'}, {'rackTotCnt': '10', 'stationName': '3918. 이좋은집오피스텔 앞', 'parkingBikeTotCnt': '11', 'shared': '110', 'stationLatitude': '37.49499130', 'stationLongitude': '126.84185028', 'stationId': 'ST-2997'}, {'rackTotCnt': '14', 'stationName': '3919. 구로역·NC신구로점 버스정류장(구로역 방면)', 'parkingBikeTotCnt': '16', 'shared': '114

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,10,3912. 삼호아파트 앞,8,80,37.49385834,126.85646820,ST-2855
1,10,3914. 신도림쌍용플래티넘노블아파트 앞,17,170,37.50126266,126.89118195,ST-2892
2,10,3915. 동선아파트 앞,14,140,37.49382782,126.83907318,ST-2949
3,10,3918. 이좋은집오피스텔 앞,11,110,37.49499130,126.84185028,ST-2997
4,14,3919. 구로역·NC신구로점 버스정류장(구로역 방면),16,114,37.50096512,126.88237000,ST-3013
...,...,...,...,...,...,...,...
722,8,6185.가양나들목,12,150,37.57341003,126.84345245,ST-3418
723,12,6187.마곡119안전센터 맞은편,4,33,37.55534744,126.82072449,ST-3415
724,10,6188.금호아파트,13,130,37.55619049,126.86463928,ST-3419
725,11,6189.데시앙플렉스 지식산업센터,13,118,37.56448364,126.84830475,ST-3424


## 1.3. 실시간 대여정보 분석

### 1.3.1.데이터 탐색 및 전처리

- 열 이름, 데이터 형식 등 열 정보를 확인합니다.

In [5]:
final.info()

<class 'pandas.DataFrame'>
RangeIndex: 2727 entries, 0 to 2726
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   rackTotCnt         2727 non-null   str  
 1   stationName        2727 non-null   str  
 2   parkingBikeTotCnt  2727 non-null   str  
 3   shared             2727 non-null   str  
 4   stationLatitude    2727 non-null   str  
 5   stationLongitude   2727 non-null   str  
 6   stationId          2727 non-null   str  
dtypes: str(7)
memory usage: 149.3 KB


- 데이터 형식을 int로 변환합니다.
- 대상 열: rackTotCnt, parkingBikeTotCnt, shared
- 구문 예) df['Money'].astype('int')

In [6]:
# 분석을 위해 승하차 인원 컬럼을 숫자형으로 변환
final[['rackTotCnt', 'parkingBikeTotCnt', 'shared']] = final[['rackTotCnt', 'parkingBikeTotCnt', 'shared']].astype(int)


- 결과를 확인합니다.

In [7]:

final.head()

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,12,80,37.55564880,126.91062927,ST-4
1,14,103. 망원역 2번출구 앞,21,150,37.55495071,126.91083527,ST-5
2,13,104. 합정역 1번출구 앞,14,108,37.55073929,126.91508484,ST-6
3,5,105. 합정역 5번출구 앞,3,60,37.55000687,126.91482544,ST-7
4,12,106. 합정역 7번출구 앞,13,108,37.54864502,126.91282654,ST-8


- 기초통계정보를 확인합니다.

In [8]:
final.describe()

,rackTotCnt,parkingBikeTotCnt,shared
count,2727.000000,2727.000000,2727.00000
mean,12.047305,12.114778,105.69747
std,5.600705,11.529144,100.31257
min,2.000000,0.000000,0.00000
25%,10.000000,4.000000,38.00000
50%,10.000000,9.000000,83.00000
75%,15.000000,17.000000,150.00000
max,65.000000,108.000000,1550.00000


### 1.3.2. 가장 많이 거치된 곳 TOP 10

- 자전거가 가장 많이 거치된 곳을 확인하기 위해 parkingBikeTotCnt 열로 내림차순 정렬한 후 상위 10개 행만 출력하세요.

In [9]:
bike_guchi_top10 = final.sort_values(by='parkingBikeTotCnt',ascending=False).head(10)
bike_guchi_top10

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
1647,62,2717.LG유플러스 마곡사옥,108,174,37.56133652,126.83390045,ST-2033
646,20,1009. 천호역4번출구(현대백화점),100,500,37.53866577,127.12424469,ST-493
2588,57,5096. LG사이언스파크 E14동,95,167,37.56349564,126.83583069,ST-3251
795,30,1210. 롯데월드타워(잠실역2번출구 쪽),86,287,37.51312637,127.10095978,ST-891
2021,15,3939.신도림역 3번출구,85,567,37.50760651,126.89134979,ST-3311
2257,10,4452. 삼성SDS,73,730,37.51633835,127.09985352,ST-2571
1910,20,3691. 둔촌역 2번출구,72,360,37.52710342,127.13651276,ST-2928
2638,30,5515.한강버스 망원 선착장,69,230,37.55466461,126.89590454,ST-3388
854,30,1291. 서울체육고등학교 앞,69,230,37.52193832,127.13186646,ST-1580
2685,20,5878. 우리은행 앞,69,345,37.52349091,126.92162323,ST-3315


### 1.3.3. 거치대 수가 가장 많은 곳 TOP 10

- 거치대 수가 가장 많은 곳을 확인하기 위해 rackTotCnt 열로 내림차순 정렬한 후 상위 10개 행만 출력하세요.

In [10]:
bike_rack_top10 = final.sort_values(by='rackTotCnt',ascending=False).head(10)
bike_rack_top10

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
1656,65,2728.마곡나루역 3번 출구,25,38,37.56580353,126.82807922,ST-2045
1647,62,2717.LG유플러스 마곡사옥,108,174,37.56133652,126.83390045,ST-2033
2588,57,5096. LG사이언스파크 E14동,95,167,37.56349564,126.83583069,ST-3251
274,51,450. 효자동 삼거리,15,29,37.58360291,126.97254944,ST-978
780,51,1193. 마곡센트럴타워 1차,28,55,37.55947113,126.83207703,ST-1688
83,46,207. 여의나루역 1번출구 앞,0,0,37.52715683,126.93190002,ST-73
1625,41,2652.송파 레미니스1단지 102동 앞,0,0,37.50883102,127.13426208,ST-1873
2584,40,5089. LG사이언스파크 E10동,45,113,37.56364822,126.83454132,ST-3128
534,40,829. 한강대교 북단 교차로,21,53,37.52293015,126.96169281,ST-1326
2591,40,5099. LG사이언스파크 E7동,41,103,37.56153488,126.83193970,ST-3260


### 1.3.4. 지도 시각화

- 위도와 경도 정보가 있으므로 지도상에 거치대 위치를 표시할 수 있습니다.
- 지도 시각화를 위해 folium 라이브러리를 설치합니다.

In [13]:
# 라이브러리 설치
# %pip install folium

- 위 분석에서 거치대가 가장 많은 곳의 좌표를 지도상에 표시합니다.

In [ ]:
import folium

m = folium.Map(location=[37.56580353, 126.82807922], tiles='OpenStreetMap', zoom_start=17)

# 마커 표시
mark1 = folium.Marker([37.56580353, 126.82807922],
                      popup='LG유플러스 마곡 사옥',
                      icon=folium.Icon(color='red')
                      )

mark1.add_to(m)

m

In [13]:
import folium

# 1. 데이터프레임의 첫 번째 행(index 0)에서 위도, 경도 추출
# 'stationLatitude'와 'stationLongitude'는 데이터프레임의 컬럼명과 일치해야 합니다.
start_lat = bike_rack_top10.iloc[0]['stationLatitude']
start_lng = bike_rack_top10.iloc[0]['stationLongitude']




m = folium.Map(location=[start_lat, start_lng], tiles='OpenStreetMap', zoom_start=17)

# 마커 표시
mark1 = folium.Marker([start_lat, start_lng],
                      popup='LG유플러스 마곡 사옥',
                      icon=folium.Icon(color='red')
                      )

mark1.add_to(m)

m

In [12]:
bike_rack_top10 = final.sort_values(by='rackTotCnt',ascending=False).head(10)
bike_rack_top10

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
1656,65,2728.마곡나루역 3번 출구,25,38,37.56580353,126.82807922,ST-2045
1647,62,2717.LG유플러스 마곡사옥,108,174,37.56133652,126.83390045,ST-2033
2588,57,5096. LG사이언스파크 E14동,95,167,37.56349564,126.83583069,ST-3251
274,51,450. 효자동 삼거리,15,29,37.58360291,126.97254944,ST-978
780,51,1193. 마곡센트럴타워 1차,28,55,37.55947113,126.83207703,ST-1688
83,46,207. 여의나루역 1번출구 앞,0,0,37.52715683,126.93190002,ST-73
1625,41,2652.송파 레미니스1단지 102동 앞,0,0,37.50883102,127.13426208,ST-1873
2584,40,5089. LG사이언스파크 E10동,45,113,37.56364822,126.83454132,ST-3128
534,40,829. 한강대교 북단 교차로,21,53,37.52293015,126.96169281,ST-1326
2591,40,5099. LG사이언스파크 E7동,41,103,37.56153488,126.83193970,ST-3260
